# Hot Days and Cold Nights

Analysis of extreme temperature exceedances at the selected GHCN station using WMO / ETCCDI-style percentile thresholds and a simpler fixed-percentile variant.

**Indicator:** annual counts (and rates) of hot days (TX90p) and cold nights (TN10p) relative to the 1961–1990 calendar-day climatology.

```{glue:figure} trend_fig
:scale: 50%
:align: right
```

**Figure. Annual number of hot days and cold nights.** Hot days exceed the 90th percentile of daily maximum temperature for that calendar day in 1961–1990; cold nights fall below the 10th percentile of daily minimum temperature for that calendar day. Solid black lines indicate statistically significant trends (p < 0.05).


## Setup

Import libraries and helper functions. Shared plotting utilities come from the [indicators_setup](https://github.com/lauracagigal/indicators_setup) repository; site configuration and percentile helpers come from `functions/air_temp.py` and `functions/temp_func.py`.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import os.path as op
import sys
from pathlib import Path

from myst_nb import glue 

import numpy as np
import pandas as pd
from datetime import datetime
from IPython.display import display

import plotly.io as pio
pio.renderers.default = "notebook_connected"

sys.path.append("../../../../../indicators_setup")
from ind_setup.plotting_int import plot_timeseries_interactive, fig_int_to_glue

sys.path.append("../../../functions")
from data_downloaders import GHCN
from temp_func import exceedance_rate_for_base_period, exceedance_rate_for_outbase_period
from air_temp import (
    load_site_config,
    site_config_filename,
    list_available_sites,
    build_output_filename,
    build_site_figures_dir,
    persist_hot_cold_outputs,
)

### Define location and variables of interest

Set `site_key` to one of the sites already configured with [`../00_site_setup.ipynb`](../00_site_setup.ipynb) (see the list printed below). The notebook loads coordinates, station ID, reference period and paths from `data/sites/<site_key>.json` — the filename is resolved with `site_config_filename()`, the same helper `00_site_setup.ipynb` uses to save it, so any site name (accents, spaces, capitals) resolves consistently.


In [2]:
sites_dir = Path('../../../data/sites')
available_sites = list_available_sites(sites_dir)
print(f"{len(available_sites)} site(s) already configured in {sites_dir}:")
display(available_sites)

1 site(s) already configured in ../../../data/sites:


,site_key,site_name,country,ghcn_station_id,ghcn_station_name,vars_interest
0,palau_psw00040309,palau_PSW00040309,Palau,PSW00040309,PW KOROR GSN 91408,"[TMIN, TMAX, PRCP]"


In [3]:
site_key = "palau_psw00040309"  # pick a site_key from the table printed above

site_config_path = sites_dir / site_config_filename(site_key)
site_cfg = load_site_config(site_config_path)

site_name = site_cfg.get('site_name', 'Site')
site_lon = float(site_cfg['site_lon'])
site_lat = float(site_cfg['site_lat'])
country = site_cfg['country']
ghcn_station_id = site_cfg['ghcn_station_id']
ghcn_station_name = site_cfg.get('ghcn_station_name', '')

data_base_dir = Path('../../../data')
site_figures_dir = build_site_figures_dir(Path('../../../outputs'), site_name, site_lon, site_lat)
data_dir = Path(data_base_dir, 'air_temp')
output_dir = data_base_dir

output_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

path_data = str(data_dir)

### Get Data / Observations from Station

Load the cached daily `TMIN` / `TMAX` pickle from `data/air_temp/GHCN_<station_id>.pkl`. All download and quality-control steps were completed in [`../00_site_setup.ipynb`](../00_site_setup.ipynb).


The data used for this analysis comes from the [GHCN-Daily](https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/) database (NOAA NCEI).

GHCN-Daily provides historical daily climate records over global land areas. Records from numerous sources are merged and subjected to quality-assurance reviews. The variables used here are **`TMAX`** (for hot days) and **`TMIN`** (for cold nights).

**Data loading:** this notebook reads the pre-processed pickle created by [`../00_site_setup.ipynb`](../00_site_setup.ipynb). It does **not** re-download from NOAA.

**Outputs:** figures → `outputs/figures/<site_tag>/`; tables and JSON → `outputs/tables/<site_tag>/`.


[GHCN-Daily documentation](https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/doc/GHCND_documentation.pdf)


In [4]:
print(f"Site: {site_name} ({country}) - GHCN station {ghcn_station_id} {ghcn_station_name}")

Site: palau_PSW00040309 (Palau) - GHCN station PSW00040309 PW KOROR GSN 91408


In [5]:
pickle_path = op.join(path_data, f'GHCN_{ghcn_station_id}.pkl')
st_data = pd.read_pickle(pickle_path)
print(f'Loaded {len(st_data)} daily records from {pickle_path}')

st_data['DATE'] = st_data.index
st_data['DAY'] = pd.to_datetime("2024-" + st_data['DATE'].dt.strftime('%m-%d'), format='%Y-%m-%d')

Loaded 25966 daily records from ../../../data/air_temp/GHCN_PSW00040309.pkl


In [6]:
st_data_daily = st_data.copy()

## Analysis

**Definitions used in this notebook:**

| Term | Criterion |
|---|---|
| Hot day (TX90p) | Daily `TMAX` exceeds the 90th percentile for that calendar day in 1961–1990 |
| Cold night (TN10p) | Daily `TMIN` falls below the 10th percentile for that calendar day in 1961–1990 |
| Fixed-percentile variant | Days above / below a single station-wide 90th / 10th percentile of the reference period |


In [7]:
exceed_rates_TMAX = exceedance_rate_for_outbase_period(st_data, "TMAX")
exceed_rates_TMIN = exceedance_rate_for_outbase_period(st_data, "TMIN")

In [8]:
TMAX_dict = dict(zip(exceed_rates_TMAX['DAY'], exceed_rates_TMAX['THRESHOLD']))
TMIN_dict = dict(zip(exceed_rates_TMIN['DAY'], exceed_rates_TMIN['THRESHOLD']))

In [9]:
df_exceed = st_data.copy()
df_exceed['THRESHOLD_TMAX'] = df_exceed['DAY'].apply(lambda day_value:TMAX_dict.get(day_value))
df_exceed['HOT_DAY'] = df_exceed[['TMAX',"THRESHOLD_TMAX"]].apply(lambda x: x["TMAX"] > x["THRESHOLD_TMAX"],axis=1)

df_exceed['THRESHOLD_TMIN'] = df_exceed['DAY'].apply(lambda day_value:TMIN_dict.get(day_value))
df_exceed['COLD_NIGHT'] = df_exceed[['TMIN',"THRESHOLD_TMIN"]].apply(lambda x: x["TMIN"] < x["THRESHOLD_TMIN"],axis=1)

df_exceed['YEAR'] = pd.DatetimeIndex(st_data['DATE']).year

In [10]:
out_of_base_hot = {}
out_of_base_cold = {}
for x in df_exceed["YEAR"].unique():
    if x > 1990:
        out_of_base_hot[x] = df_exceed[df_exceed["YEAR"] == x]['HOT_DAY'].mean()
        out_of_base_cold[x] = df_exceed[df_exceed["YEAR"] == x]['COLD_NIGHT'].mean()

Calendar-day percentile thresholds are estimated for the out-of-base and base periods. Annual hot-day and cold-night rates (and counts) are then derived for each year in the record.


In [11]:
ex_cold, all_cold = exceedance_rate_for_base_period(st_data, "TMIN")
ex_hot, all_hot = exceedance_rate_for_base_period(st_data, "TMAX")
all_hot = ex_hot|out_of_base_hot
all_cold = ex_cold|out_of_base_cold
cold_bar = sum(ex_cold.values()) / len(ex_cold)
hot_bar = sum(ex_hot.values()) / len(ex_hot)

In [12]:
hot_anom = {}

for x in all_hot:
    hot_anom[x] = 100*(all_hot[x]-hot_bar)

cold_anom = {}
for x in all_cold:
    cold_anom[x] = 100*(all_cold[x]-cold_bar)

In [13]:
df_cold_anom = pd.DataFrame.from_dict(cold_anom, orient='index', columns=['Perc_Anom'])
df_cold_anom.index = pd.to_datetime(df_cold_anom.index, format='%Y')

df_hot_anom = pd.DataFrame.from_dict(hot_anom, orient='index', columns=['Perc_Anom'])
df_hot_anom.index = pd.to_datetime(df_hot_anom.index, format='%Y')

### Plotting


#### Cold nights (TN10p)

Annual cold-night counts with a fitted linear trend.


In [14]:
dict_plot = [{'data' : df_cold_anom*3.6525, 'var' : 'Perc_Anom', 'ax' : 1, 'label' : 'Cold Nights'},]
fig, TRENDS = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12), return_trend = True, label_yaxes = 'Number of days/year')

#### Hot days (TX90p)

Annual hot-day counts with a fitted linear trend.


In [15]:
dict_plot = [{'data' : df_hot_anom*3.6525, 'var' : 'Perc_Anom', 'ax' : 1, 'label' : 'Hot Days', 'color':'red'}]
fig, TRENDS = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12), return_trend = True, label_yaxes = 'Number of days/year')

#### Hot days and cold nights together


Combined annual counts of hot days and cold nights relative to the 1961–1990 calendar-day percentiles. Saved as `F4_ST_hot_cold_<site_tag>.html` / `.png`.


In [16]:
dict_plot = [{'data' : df_cold_anom*3.6525, 'var' : 'Perc_Anom', 'ax' : 1, 'label' : 'Cold Nights'},
             {'data' : df_hot_anom*3.6525, 'var' : 'Perc_Anom', 'ax' : 1, 'label' : 'Hot Days'}]
fig, TRENDS = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12), return_trend = True, label_yaxes = 'Number of days/year')
fig.write_html(
    site_figures_dir / build_output_filename('F4_ST_hot_cold', site_name, site_lon, site_lat, ext='html'),
    include_plotlyjs="cdn",
)
fig.write_image(
    site_figures_dir / build_output_filename('F4_ST_hot_cold', site_name, site_lon, site_lat),
    width=1200, height=600,
)
glue("trend_cold_decade", float(TRENDS[0]*10), display=False)
glue("trend_hot_decade", float(TRENDS[1]*10), display=False)
glue("trend_fig", fig_int_to_glue(fig), display=False)


**Figure.** Annual number of hot days and cold nights relative to 1961–1990 climatology. Solid black lines indicate statistically significant trends (p < 0.05).


The plot below shows the same information expressed as a percentage of days in the year. Saved as `F4_ST_hot_cold_percentiles_<site_tag>.html` / `.png`.


In [17]:
dict_plot = [{'data' : df_cold_anom, 'var' : 'Perc_Anom', 'ax' : 1, 'label' : 'Cold Nights'},
             {'data' : df_hot_anom, 'var' : 'Perc_Anom', 'ax' : 1, 'label' : 'Hot Days'}]
fig, _ = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12), return_trend = True, label_yaxes = '% of days/year')


In [18]:
annual_cold = df_cold_anom*3.6525
annual_hot = df_hot_anom*3.6525
annual_cold['Perc_Anom'] = np.where(annual_cold['Perc_Anom'] > 0, annual_cold['Perc_Anom'], annual_cold['Perc_Anom'])
annual_hot['Perc_Anom'] = np.where(annual_hot['Perc_Anom'] > 0, annual_hot['Perc_Anom'], annual_hot['Perc_Anom'])

### Summary table and persisted outputs

The following cells build ETCCDI-style and fixed-percentile summary tables and call `persist_hot_cold_outputs()` to save:

- `T_hot_days_per_year_<site_tag>.csv`
- `T_cold_nights_per_year_<site_tag>.csv`
- `T_hot_cold_summary_table_etccdi_<site_tag>.csv`
- `T_hot_cold_summary_table_percentiles_<site_tag>.csv`
- `T_hot_cold_summary_metrics_<site_tag>.json`


In [19]:
from ind_setup.tables import table_temp_13, table_temp_13b

In [20]:
summary_table_etccdi = table_temp_13(st_data, annual_hot, annual_cold, df_hot_anom, df_cold_anom, TRENDS)

### Fixed-percentile variant


Alternative definition using a single station-wide 90th percentile of `TMAX` and 10th percentile of `TMIN` over the reference period (not calendar-day specific). Useful as a simpler companion metric to TX90p / TN10p.


In [21]:
q90 = st_data.loc['1961':'1991'].TMAX.quantile(0.9)
q10 = st_data.loc['1961':'1991'].TMIN.quantile(0.1)
print('The 10th percentile of TMIN in the period 1961-1991 is:', q10)
print('The 90th percentile of TMAX in the period 1961-1991 is:', q90)

The 10th percentile of TMIN in the period 1961-1991 is: 23.3
The 90th percentile of TMAX in the period 1961-1991 is: 32.2


In [22]:
st_data_min = st_data[['TMIN']]
# st_data_min['TMIN'] = st_data_min['TMIN'] + .5
st_data_min = st_data_min.loc[st_data_min['TMIN'] < q10]
st_min_counts = st_data_min.groupby(st_data_min.index.year).count()
st_min_counts.index = pd.to_datetime(st_min_counts.index, format='%Y')
print(f'The average number of cold nights per year in the period 1961-1991 is: {st_min_counts.loc["1961":"1991"].mean().values[0]:.0f}')

st_data_max = st_data[['TMAX']]
# st_data_max['TMAX'] = st_data_max['TMAX'] + .5
st_data_max = st_data_max.loc[st_data_max['TMAX'] > q90]
st_max_counts = st_data_max.groupby(st_data_max.index.year).count()
st_max_counts.index = pd.to_datetime(st_max_counts.index, format='%Y')
print(f'The average number of hot days per year in the period 1961-1991 is: {st_max_counts.loc["1961":"1991"].mean().values[0]:.0f}')

The average number of cold nights per year in the period 1961-1991 is: 32
The average number of hot days per year in the period 1961-1991 is: 17


In [23]:
dict_plot = [{'data' : st_min_counts, 'var' : 'TMIN', 'ax' : 1, 'label' : 'Cold Nights below 10th percentile'},
             {'data' : st_max_counts, 'var' : 'TMAX', 'ax' : 1, 'label' : 'Hot Days over 90th percentile'}]
fig, TRENDS_prctiles = plot_timeseries_interactive(dict_plot, trendline=True, figsize = (25, 12), return_trend = True, label_yaxes = 'Number of days/year')
fig.write_html(
    site_figures_dir / build_output_filename('F4_ST_hot_cold_percentiles', site_name, site_lon, site_lat, ext='html'),
    include_plotlyjs="cdn",
)
fig.write_image(
    site_figures_dir / build_output_filename('F4_ST_hot_cold_percentiles', site_name, site_lon, site_lat),
    width=1200, height=600,
)


In [24]:
summary_table_percentiles = table_temp_13b(st_data, st_max_counts, st_min_counts, TRENDS_prctiles)
persist_hot_cold_outputs(
    Path('../../../outputs'),
    site_name, site_lon, site_lat,
    ghcn_station_id, ghcn_station_name, country,
    df_exceed, annual_hot, annual_cold,
    summary_table_etccdi, summary_table_percentiles,
    TRENDS, TRENDS_prctiles,
    st_max_counts, st_min_counts, q90, q10,
)

Metric,Value,Year
Daily Maximum Temperature (°C),35.000,
Daily Minimum Temperature (°C),20.600,
,,
Average number of hot days,24.831,
Change in Average Annual Number of Hot Days,0.956,
Average Annual Number of Hot days: 1961-1971,-9.379,
Average Annual Number of Hot days: 2001-2011,67.099,
Average Annual Number of Hot days: 2011-2021,33.136,
Maximum number of hot days,162.000,1998
Minimum number of hot days,-17.000,2021


Metric,Value,Year
Daily Maximum Temperature (°C),35.000,
Daily Minimum Temperature (°C),20.600,
,,
Average number of hot days,36.265,
Change in Average Annual Number of Hot Days,1.126,
Average Annual Number of Hot days: 1961-1971,8.600,
Average Annual Number of Hot days: 2001-2011,72.727,
Average Annual Number of Hot days: 2011-2021,46.889,
Maximum number of hot days,164.000,1998
Minimum number of hot days,1.000,1956


PosixPath('../../../outputs/tables/palau_psw00040309_lat7p337_lon134p477')